# 第 36 章 · benchmark 与正确性：测准、对齐、再上线

本章回答：怎么测一个 kernel 的时间才算数？有效带宽怎么算、怎么判断离上限多远？一个新 kernel 要怎么测才敢替换原来的实现？`assert_close` 的容差是多少，低精度时怎么设？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/36-bench-correctness.md`（讲解、动画、练习都在讲义里）。对应原文：官方文档 `torch.utils.benchmark`、`torch.testing.assert_close`、Reproducibility（〔在线〕）；Triton `triton.testing.do_bench`（〔在线〕）；AISPE ch13「持续集成和性能基准测试」（`ch13.md` 第 1620–1691 行）；本讲义 [第 5 章](05-cuda-semantics.md)、[第 29 章](29-why-fusion.md)；本章计时和容差都是本机 torch 2.15.0 实测

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import time

from torch.utils import benchmark
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 四个坑 | 热身、中位数、线程数 | `naive_timings()`、`timer_median_ms()` | 首次约 2 ms vs 中位约 0.42 ms；Timer 1 线程约 0.70 ms |
| 第二步 有效带宽 | 50.3 MB ÷ 0.70 ms | `effective_gbs(...)` | 72 GB/s |
| 第三步 容差 | atol + rtol·|b| | `tolerance_checks()` | fp32 1e-5 过、1e-3 不过；bf16 0.016 过、0.03 不过 |
| 第四步 不连续 bug | 按 storage 读 | `noncontiguous_bug()` | [0, 3, 1, 4, 2, 5] vs [0, 1, 2, 3, 4, 5] |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `effective_gbs`**（来自 `ch36_bench_correctness.py`）：下面要用到的小工具。

In [ ]:
def effective_gbs(nbytes: float, ms: float) -> float:
    return nbytes / ms / 1e6

**函数 `naive_timings`**（来自 `ch36_bench_correctness.py`）：下面要用到的小工具。

In [ ]:
def naive_timings(n: int = 1 << 22, reps: int = 30):
    x, y = torch.randn(n), torch.randn(n)
    t = time.perf_counter()
    torch.relu(x + y)
    first = (time.perf_counter() - t) * 1e3
    ts = []
    for _ in range(reps):
        t = time.perf_counter()
        torch.relu(x + y)
        ts.append((time.perf_counter() - t) * 1e3)
    ts.sort()
    return first, ts[len(ts) // 2]

**函数 `noncontiguous_bug`**（来自 `ch36_bench_correctness.py`）：下面要用到的小工具。

In [ ]:
def noncontiguous_bug():
    t = torch.arange(6.0).view(2, 3).t()
    return t.reshape(-1).tolist(), torch.as_strided(t, (6,), (1,)).tolist()

**函数 `timer_median_ms`**（来自 `ch36_bench_correctness.py`）：下面要用到的小工具。

In [ ]:
def timer_median_ms(n: int = 1 << 22, threads: int = 1) -> float:
    x, y = torch.randn(n), torch.randn(n)
    m = benchmark.Timer("torch.relu(x + y)", globals=dict(x=x, y=y, torch=torch), num_threads=threads)
    return m.blocked_autorange(min_run_time=0.5).median * 1e3

**函数 `tolerance_checks`**（来自 `ch36_bench_correctness.py`）：在 1.0 上加 d，assert_close 是否通过。

In [ ]:
def tolerance_checks():
    """在 1.0 上加 d，assert_close 是否通过。"""
    res = {}
    for dt, ds in ((torch.float32, (1e-7, 1e-5, 1e-3)), (torch.bfloat16, (0.0078, 0.016, 0.03))):
        b = torch.tensor([1.0], dtype=dt)
        for d in ds:
            a = (b.float() + d).to(dt)
            try:
                torch.testing.assert_close(a, b)
                res[(str(dt), d)] = True
            except AssertionError:
                res[(str(dt), d)] = False
    return res

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 自己掐表 (首次 ms, 中位数 ms)：", naive_timings())
print("benchmark.Timer 默认 1 线程中位数", round(timer_median_ms(), 2), "ms；用全部线程", round(timer_median_ms(threads=torch.get_num_threads()), 2), "ms")

In [ ]:
print("\n第二步 50.3 MB / 0.70 ms ≈", round(effective_gbs(3 * (1 << 22) * 4, 0.70)), "GB/s")

In [ ]:
print("\n第三步 容差：", tolerance_checks())

In [ ]:
print("\n第四步 不连续输入 正确 vs 按 storage 读：", noncontiguous_bug())

## 3. 代码里才看得到的

1. **`benchmark.Timer` 默认 `num_threads=1`**：和直接跑（多线程）比较时要设成一样。

## 4. 自己改一改

- `timer_median_ms(threads=2)`、`threads=4`，看线程数和时间的关系。
- 在 `tolerance_checks` 里加 float16，看默认容差。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 测时间：先热身、要同步、多次取中位、线程数一致，GPU 用 do_bench；
> 有效带宽是必须字节除以时间，和峰值比才知好坏；
> 对数值：assert_close 按 dtype 定容差，参考用高精度、反向也要比；
> 测试矩阵别偷懒：不整除、不连续、边界值、compile 里也要跑；发现、判断、写、注册、验、测、接、守。